# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, strictly referencing dataset entities by their `@id`.

### Dataset Source
The dataset is described via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset title: ", metadata.name)
print("Description: ", metadata.description)
print("---")
print("Data Use Cases:")
for use_case in metadata.dataUseCases:
    print(" -", use_case)
print("---")
print("Keywords:", ', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else None)
print("Published:", metadata.datePublished)


## 2. Data Overview
Review available record sets, fields, and their `@id`s. This step helps identify the structure of the dataset and which record sets and fields are available for extraction and processing.

In [ ]:
# List all available RecordSets by @id
print("Record Sets (by @id):\n")
for rs in dataset.record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', None)}")

# For each RecordSet, list its fields (columns) and their @id
record_sets_overview = {}
for rs in dataset.record_sets:
    field_ids = []
    print("\nRecordSet:", rs.get('name', None), f"(@id={rs['@id']})")
    if 'field' in rs and rs['field']:
        # field can be a list of @id, or a single dict
        if isinstance(rs['field'], list):
            field_ids = rs['field']
        else:
            field_ids = [rs['field']]
        print("  Fields/Columns:")
        for field_id in field_ids:
            # Look up the field object by its @id in dataset.fields
            field_obj = next((f for f in dataset.fields if f['@id'] == field_id), None)
            print(f"    @id: {field_id} | name: {field_obj.get('name', None) if field_obj else None}")
    else:
        print("  (No fields listed)")
    record_sets_overview[rs['@id']] = field_ids

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Each record set and field is referenced by its `@id`.

In [ ]:
# Prepare to extract data from record sets by their @id
# Let's use all available record sets
record_sets = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Preview the data for each record set
for record_set_id in record_sets:
    print(f"---\nExtracting records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    display(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data cleaning and processing, such as filtering, normalization, and grouping by attributes, always referencing fields by `@id`.

Below, we demonstrate EDA using the main record set (the first record set found).

In [ ]:
import numpy as np

# Pick the first record set (for demonstration)
main_record_set_id = record_sets[0]
df = dataframes[main_record_set_id]

# Show columns
print("Available columns:", df.columns.tolist())

# Select a numeric field for demonstration. Adjust the field '@id' as needed from data overview
numeric_field_id = None
for col in df.columns:
    # Heuristically pick a column with likely numeric data
    if any(x in col.lower() for x in ['abundance', 'count', 'percentage', 'weight', 'coverage']):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Fallback to the first float-like column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if numeric_field_id is None:
    print("No obvious numeric field found for EDA.")
else:
    # Filter: Keep records with the value > threshold (e.g., mean)
    threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize this field (z-score normalization)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another attribute (e.g., 'description' or another categorical field by @id)
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == 'O':
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(
            f"mean_{numeric_field_id}")
        print(f"\nGrouped by '{group_field}', mean of '{numeric_field_id}':")
        display(grouped_df.head())

## 5. Visualization
Visualize numeric value distributions and field relationships. Here, a histogram for the selected numeric field and a boxplot by group/categorical attribute is produced, referencing each by its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        # For clarity, limit the number of groups
        n_groups = df[group_field].value_counts().index[:10]
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df[df[group_field].isin(n_groups)])
        plt.xticks(rotation=45)
        plt.title(f"'{numeric_field_id}' by '{group_field}' (@id)")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and process the FAIR^2 human protein mass spectrometry dataset using `mlcroissant`.

- All dataset entities (record sets, fields) were referenced by their Croissant `@id`.
- Tabular data records were loaded into pandas DataFrames.
- Basic EDA steps (filtering, normalization, grouping) were performed on selected fields.
- Data distributions and category relationships were visualized.

This notebook can serve as a template for working with other Croissant-based datasets using the `mlcroissant` library.